# 🚀 TCN (Temporal Convolutional Network) — KRW-BTC 1분봉 이진 분류

**Step DL-5 Colab 학습 노트북**

| 항목 | 내용 |
|---|---|
| 모델 | 1D-CNN (TCN) — dilations=[1,2,4,8,16,32] |
| 입력 | (Batch, SEQ_LEN=60, Features=52) |
| 출력 | 이진 logit → sigmoid → prob |
| 목표 | Precision 최우선 (현물 롱 온리 전략) |
| 수용 영역 | 253 timesteps ≈ 4.2시간 |

---
### 실행 순서
1. **런타임 → GPU로 변경** (런타임 → 런타임 유형 변경 → T4 GPU)
2. 셀 순서대로 실행 (`Shift+Enter`)
3. 학습 완료 후 모델이 Google Drive에 자동 저장

## 1️⃣ 환경 설정

In [ ]:
# 필수 패키지 설치
!pip install pyarrow fastparquet scikit-learn --quiet
print('설치 완료')

In [ ]:
import torch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'디바이스: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'메모리: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  GPU 없음 — 런타임 유형을 GPU로 변경하세요')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive 마운트 완료')

## 2️⃣ 경로 및 하이퍼파라미터 설정

In [ ]:
from pathlib import Path

# ── Google Drive 경로 설정 ──────────────────────────────────────────────
# 아래 경로를 실제 Drive 업로드 위치에 맞게 수정하세요
DRIVE_ROOT = Path('/content/drive/MyDrive/btc_quant')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

# 데이터 파일 경로 (Drive에 업로드한 parquet 파일)
DATA_PATH = DRIVE_ROOT / 'btc_1m_dl_2y.parquet'   # 2년치
# DATA_PATH = DRIVE_ROOT / 'btc_1m_dl.parquet'    # 6개월치 (대안)

# 아티팩트 저장 경로
ARTIFACT_DIR = DRIVE_ROOT / 'artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

TCN_MODEL_PATH   = ARTIFACT_DIR / 'tcn_model.pt'
TCN_META_PATH    = ARTIFACT_DIR / 'tcn_model_meta.json'
SCALER_PATH      = ARTIFACT_DIR / 'scaler.joblib'
FEATURE_COLS_PATH = ARTIFACT_DIR / 'feature_cols.json'

print(f'데이터 경로: {DATA_PATH}')
print(f'아티팩트 저장: {ARTIFACT_DIR}')
print(f'파일 존재 여부: {DATA_PATH.exists()}')

# ── 하이퍼파라미터 ──────────────────────────────────────────────────────
SEQ_LEN          = 60
BATCH_SIZE       = 512
TRAIN_STRIDE     = 2
N_CHANNELS       = 64
KERNEL_SIZE      = 3
DROPOUT          = 0.2
LR               = 2e-4
WEIGHT_DECAY     = 1e-3
MAX_EPOCHS       = 60
ES_PATIENCE      = 12
LR_PATIENCE      = 5
LR_FACTOR        = 0.5
POS_WEIGHT_CAP   = 1.0
MIN_RECALL_FLOOR = 0.05
THRESHOLD        = 0.55

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15

EXCLUDE_COLS = {'target', 'future_ret'}

print('설정 완료')

## 3️⃣ 라이브러리 임포트

In [ ]:
import json
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (
    confusion_matrix, f1_score, precision_score,
    recall_score, roc_auc_score,
)
from sklearn.preprocessing import RobustScaler
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings('ignore')
print('임포트 완료')

## 4️⃣ 데이터 로드 및 전처리

In [ ]:
# ── CryptoTimeSeriesDataset ─────────────────────────────────────────────
class CryptoTimeSeriesDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray, seq_len: int = SEQ_LEN, stride: int = 1):
        assert len(X) > seq_len
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.seq_len = seq_len
        self._indices = list(range(0, len(X) - seq_len, max(1, stride)))

    def __len__(self): return len(self._indices)

    def __getitem__(self, idx):
        s = self._indices[idx]
        return self.X[s:s + self.seq_len], self.y[s + self.seq_len]


# ── 데이터 로드 ─────────────────────────────────────────────────────────
print(f'[Data] 로드 중: {DATA_PATH.name}')
df_raw = pd.read_parquet(DATA_PATH)
df_raw = df_raw.dropna(subset=['target'])

feature_cols = [c for c in df_raw.columns if c not in EXCLUDE_COLS]
df_raw[feature_cols] = df_raw[feature_cols].ffill().bfill()

n = len(df_raw)
n_train = int(n * TRAIN_RATIO)
n_val   = int(n * VAL_RATIO)

train_df = df_raw.iloc[:n_train]
val_df   = df_raw.iloc[n_train:n_train + n_val]
test_df  = df_raw.iloc[n_train + n_val:]

print(f'전체 유효 행: {n:,}')
print(f'Train: {len(train_df):,}  ({train_df.index[0].date()} ~ {train_df.index[-1].date()})')
print(f'Val:   {len(val_df):,}  ({val_df.index[0].date()} ~ {val_df.index[-1].date()})')
print(f'Test:  {len(test_df):,}  ({test_df.index[0].date()} ~ {test_df.index[-1].date()})')
print(f'피처 수: {len(feature_cols)}')

# ── 스케일러 fit (Train only) ────────────────────────────────────────────
scaler = RobustScaler()
X_train = scaler.fit_transform(train_df[feature_cols].values).astype(np.float32)
X_val   = scaler.transform(val_df[feature_cols].values).astype(np.float32)
X_test  = scaler.transform(test_df[feature_cols].values).astype(np.float32)

y_train = train_df['target'].values.astype(np.float32)
y_val   = val_df['target'].values.astype(np.float32)
y_test  = test_df['target'].values.astype(np.float32)

# ── 아티팩트 저장 ────────────────────────────────────────────────────────
joblib.dump(scaler, SCALER_PATH)
with open(FEATURE_COLS_PATH, 'w') as f:
    json.dump(feature_cols, f, ensure_ascii=False, indent=2)
print(f'스케일러 저장: {SCALER_PATH}')
print(f'피처 목록 저장: {FEATURE_COLS_PATH}')

# ── DataLoader ────────────────────────────────────────────────────────────
train_ds = CryptoTimeSeriesDataset(X_train, y_train, SEQ_LEN, stride=TRAIN_STRIDE)
val_ds   = CryptoTimeSeriesDataset(X_val,   y_val,   SEQ_LEN, stride=1)
test_ds  = CryptoTimeSeriesDataset(X_test,  y_test,  SEQ_LEN, stride=1)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train 샘플: {len(train_ds):,} / Val: {len(val_ds):,} / Test: {len(test_ds):,}')
print(f'Train LONG 비율: {y_train.mean():.4f}')

## 5️⃣ TCN 모델 정의

In [ ]:
# ── TCN 빌딩 블록 ────────────────────────────────────────────────────────
class _Chomp1d(nn.Module):
    def __init__(self, chomp_size): super().__init__(); self.c = chomp_size
    def forward(self, x): return x[:, :, :-self.c].contiguous()


class _TemporalBlock(nn.Module):
    def __init__(self, n_in, n_out, ks, dilation, dropout=0.2):
        super().__init__()
        pad = (ks - 1) * dilation
        self.net = nn.Sequential(
            nn.utils.weight_norm(nn.Conv1d(n_in, n_out, ks, padding=pad, dilation=dilation)),
            _Chomp1d(pad), nn.ReLU(), nn.Dropout(dropout),
            nn.utils.weight_norm(nn.Conv1d(n_out, n_out, ks, padding=pad, dilation=dilation)),
            _Chomp1d(pad), nn.ReLU(), nn.Dropout(dropout),
        )
        self.downsample = nn.Conv1d(n_in, n_out, 1) if n_in != n_out else None
        self.relu = nn.ReLU()

    def forward(self, x):
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(self.net(x) + res)


class TCNClassifier(nn.Module):
    """Temporal Convolutional Network 이진 분류기.
    Input : (B, T, F)  →  Output: (B, 1) logit
    Receptive Field: 253 timesteps = 4.2시간
    """
    DILATIONS = [1, 2, 4, 8, 16, 32]

    def __init__(self, n_features, n_channels=64, kernel_size=3, dropout=0.2):
        super().__init__()
        self.n_features  = n_features
        self.n_channels  = n_channels
        self.kernel_size = kernel_size
        self.dropout     = dropout

        blocks = []
        for i, d in enumerate(self.DILATIONS):
            in_ch = n_features if i == 0 else n_channels
            blocks.append(_TemporalBlock(in_ch, n_channels, kernel_size, d, dropout))
        self.tcn = nn.Sequential(*blocks)

        self.head = nn.Sequential(
            nn.Linear(n_channels, n_channels // 2),
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(n_channels // 2, 1),
        )

    def forward(self, x):           # x: (B, T, F)
        x = x.transpose(1, 2)       # → (B, F, T)
        x = self.tcn(x)             # → (B, C, T)
        return self.head(x[:, :, -1])  # → (B, 1)

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def save(self, model_path, meta_path):
        torch.save(self.state_dict(), model_path)
        with open(meta_path, 'w') as f:
            json.dump({'n_features': self.n_features, 'n_channels': self.n_channels,
                       'kernel_size': self.kernel_size, 'dropout': self.dropout}, f, indent=2)


# ── 모델 초기화 ─────────────────────────────────────────────────────────
n_features = len(feature_cols)
model = TCNClassifier(
    n_features=n_features,
    n_channels=N_CHANNELS,
    kernel_size=KERNEL_SIZE,
    dropout=DROPOUT,
).to(DEVICE)

rf = 1 + 2 * (KERNEL_SIZE - 1) * sum(TCNClassifier.DILATIONS)
print(f'TCN 파라미터: {model.count_params():,}')
print(f'수용 영역: {rf} timesteps ({rf/60:.2f}시간)')
print(f'디바이스: {DEVICE}')

# shape 검증
with torch.no_grad():
    _dummy = torch.randn(2, SEQ_LEN, n_features).to(DEVICE)
    _out = model(_dummy)
    assert _out.shape == (2, 1)
print(f'출력 shape 검증: {tuple(_out.shape)}  ✓')

## 6️⃣ 학습

In [ ]:
# ── 손실함수 ────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, alpha=1.0, gamma=2.0, pos_weight=None):
        super().__init__()
        self.alpha = alpha; self.gamma = gamma
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction='none')

    def forward(self, inp, tgt):
        bce = self.bce(inp, tgt)
        pt  = torch.exp(-bce)
        return (self.alpha * (1 - pt) ** self.gamma * bce).mean()


n_long = y_train.sum()
n_flat = len(y_train) - n_long
pw_val = min(n_flat / max(n_long, 1), POS_WEIGHT_CAP)
pw = torch.tensor([pw_val], dtype=torch.float32, device=DEVICE)
print(f'pos_weight = {pw_val:.4f}  (POS_WEIGHT_CAP={POS_WEIGHT_CAP})')

criterion = FocalLoss(alpha=1.0, gamma=2.0, pos_weight=pw).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=LR_PATIENCE, factor=LR_FACTOR
)


def train_epoch(model, loader, optimizer, criterion):
    model.train(); total = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE).unsqueeze(1)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def evaluate(model, loader, criterion, threshold=0.55):
    model.eval(); total = 0.0; all_p, all_l = [], []
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE).unsqueeze(1)
        logit = model(xb)
        total += criterion(logit, yb).item()
        all_p.append(torch.sigmoid(logit).squeeze(1).cpu().numpy())
        all_l.append(yb.squeeze(1).cpu().numpy())
    probs  = np.concatenate(all_p)
    labels = np.concatenate(all_l)
    preds  = (probs >= threshold).astype(int)
    return {
        'loss':      total / len(loader),
        'auc':       roc_auc_score(labels, probs),
        'precision': precision_score(labels, preds, zero_division=0),
        'recall':    recall_score(labels, preds, zero_division=0),
        'f1':        f1_score(labels, preds, zero_division=0),
        'probs': probs, 'labels': labels,
    }

print('학습 유틸 준비 완료')

In [ ]:
# ── 학습 루프 ────────────────────────────────────────────────────────────
best_precision = 0.0
best_loss      = float('inf')
best_epoch     = 0
es_counter     = 0
t_start        = time.time()

print(f'{'Ep':>4} | {'TrLoss':>7} | {'VaLoss':>7} | {'VaAUC':>7} | '
      f'{'VaPrec':>7} | {'VaRec':>7} | {'LR':>8} | {'s':>5}')
print('-' * 72)

for epoch in range(1, MAX_EPOCHS + 1):
    t_ep = time.time()
    tr_loss = train_epoch(model, train_loader, optimizer, criterion)
    vm      = evaluate(model, val_loader, criterion, threshold=THRESHOLD)

    scheduler.step(vm['loss'])
    cur_lr  = optimizer.param_groups[0]['lr']
    elapsed = time.time() - t_ep

    # Precision 우선 Early Stopping
    improved = False
    if vm['recall'] >= MIN_RECALL_FLOOR and vm['precision'] > best_precision:
        best_precision = vm['precision']
        best_epoch = epoch; es_counter = 0
        model.save(TCN_MODEL_PATH, TCN_META_PATH)
        improved = True
    elif vm['loss'] < best_loss and best_precision == 0.0:
        best_loss = vm['loss']
        best_epoch = epoch; es_counter = 0
        model.save(TCN_MODEL_PATH, TCN_META_PATH)
        improved = True
    else:
        es_counter += 1

    mark = ' ★' if improved else ''
    print(f'{epoch:4d} | {tr_loss:7.4f} | {vm["loss"]:7.4f} | {vm["auc"]:7.4f} | '
          f'{vm["precision"]:7.4f} | {vm["recall"]:7.4f} | {cur_lr:8.2e} | {elapsed:4.1f}s{mark}')

    if es_counter >= ES_PATIENCE:
        print(f'\n⏹  Early Stopping (epoch={epoch}, patience={ES_PATIENCE})')
        break

total_min = (time.time() - t_start) / 60
print(f'\n학습 완료  Best epoch={best_epoch}  총 {total_min:.1f}분')

## 7️⃣ 평가 및 Threshold 탐색

In [ ]:
# ── 테스트 평가 ─────────────────────────────────────────────────────────
# Best 모델 다시 로드 (weight_norm은 state_dict로 자동 처리)
with open(TCN_META_PATH) as f:
    meta = json.load(f)
best_model = TCNClassifier(**meta).to(DEVICE)
best_model.load_state_dict(torch.load(TCN_MODEL_PATH, map_location=DEVICE))
best_model.eval()

tm = evaluate(best_model, test_loader, criterion, threshold=THRESHOLD)
print(f'Test AUC-ROC  : {tm["auc"]:.4f}')
print(f'Test Precision: {tm["precision"]:.4f}  ← 핵심 지표')
print(f'Test Recall   : {tm["recall"]:.4f}')
print(f'Test F1       : {tm["f1"]:.4f}')

# 혼동 행렬
preds_055 = (tm['probs'] >= THRESHOLD).astype(int)
cm = confusion_matrix(tm['labels'], preds_055)
tn, fp, fn, tp = cm.ravel()
print(f'\n혼동 행렬 (threshold={THRESHOLD})')
print(f'  TN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}')
print(f'  잘못된 진입 비율: {fp/(tp+fp+1e-9):.4f}')

In [ ]:
# ── Threshold 탐색 ───────────────────────────────────────────────────────
print(f'{'Threshold':>10}  {'Precision':>10}  {'Recall':>8}  {'F1':>8}  {'N_LONG':>8}')
print('-' * 55)

best_thr = {'threshold': THRESHOLD, 'precision': 0.0}
for thr in [round(t, 2) for t in np.arange(0.45, 0.76, 0.05)]:
    preds = (tm['probs'] >= thr).astype(int)
    prec  = precision_score(tm['labels'], preds, zero_division=0)
    rec   = recall_score(tm['labels'], preds, zero_division=0)
    f1    = f1_score(tm['labels'], preds, zero_division=0)
    n_p   = int(preds.sum())
    mark  = ''
    if rec >= MIN_RECALL_FLOOR and prec > best_thr['precision']:
        best_thr = {'threshold': thr, 'precision': prec, 'recall': rec}
        mark = ' ← best'
    print(f'{thr:>10.2f}  {prec:>10.4f}  {rec:>8.4f}  {f1:>8.4f}  {n_p:>8,}{mark}')

print(f'\n최적 Threshold: {best_thr["threshold"]}  Precision={best_thr["precision"]:.4f}')

## 8️⃣ Drive 저장 및 다운로드

In [ ]:
# 아티팩트는 이미 Drive에 저장됨 (학습 중 save() 호출)
print('Drive 저장된 파일:')
for f in ARTIFACT_DIR.glob('*'):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:30s}  {size_kb:8.1f} KB')

# 결과 요약 JSON 저장
result = {
    'best_epoch': best_epoch,
    'test_auc': round(float(tm['auc']), 4),
    'test_precision': round(float(tm['precision']), 4),
    'test_recall': round(float(tm['recall']), 4),
    'test_f1': round(float(tm['f1']), 4),
    'best_threshold': best_thr['threshold'],
    'best_threshold_precision': round(float(best_thr['precision']), 4),
    'tcn_params': best_model.count_params(),
    'receptive_field': 1 + 2 * (KERNEL_SIZE - 1) * sum(TCNClassifier.DILATIONS),
    'data_rows': n,
    'train_period': f'{train_df.index[0].date()} ~ {train_df.index[-1].date()}',
    'test_period':  f'{test_df.index[0].date()} ~ {test_df.index[-1].date()}',
}
result_path = ARTIFACT_DIR / 'tcn_train_result.json'
with open(result_path, 'w') as f:
    json.dump(result, f, indent=2, ensure_ascii=False)

print(f'\n결과 요약 저장: {result_path}')
print(json.dumps(result, indent=2, ensure_ascii=False))

In [ ]:
# ── 로컬 다운로드 (서버 복사용) ─────────────────────────────────────────
from google.colab import files

print('아래 파일들을 다운로드하여 artifacts/dl_prod/ 에 배치하세요.')
for f in [TCN_MODEL_PATH, TCN_META_PATH, SCALER_PATH, FEATURE_COLS_PATH, result_path]:
    if f.exists():
        print(f'  다운로드: {f.name}')
        files.download(str(f))